# The associations between polypharmacy and individual PGS

## Notes before running the scripts
- Remember to replace all data and working directory paths with your actual paths.
- Dependency: haven, tidyverse, broom, data.table, tidyr, glue, writexl

In [ ]:
library(haven)
library(tidyverse)
library(broom)
library(tidyr)
library(parallel)
library(data.table)
library(glue)
library(writexl)

In [ ]:
safe_formula <- function(phenotype, predictors) {
  # Wrap predictors in backticks if they start with a number or contain special chars
  safe_preds <- sapply(predictors, function(x) {
    if (grepl("^[0-9]|[^a-zA-Z0-9_\\.]", x)) {
      paste0("`", x, "`")
    } else {
      x
    }
  })
  
  as.formula(paste(phenotype, "~", paste(safe_preds, collapse = " + ")))
}

In [ ]:
model2logistic <- function(data, predictor, phenotype, covariates){
    
    data$Sex <- factor(data$Sex)

    scaled_predictor <- predictor
    scaled_covariates <- covariates
    
    scale_vars <- unique(c(predictor, covariates))
    
    for(v in scale_vars){
      if(v == "Sex" || is.factor(data[[v]])) next
      
      if(is.numeric(data[[v]]) && length(unique(na.omit(data[[v]]))) > 2){
        
        new_col_name <- paste0(v, "_scaled")
        
        data[[new_col_name]] <- as.numeric(scale(data[[v]]))
        
        if(v == predictor) {
          scaled_predictor <- new_col_name
        }
        if(v %in% covariates) {
          scaled_covariates[scaled_covariates == v] <- new_col_name
        }
      }
    }
    
    pred_cov <- c(scaled_predictor, scaled_covariates, "Sex")
    formula <- safe_formula(phenotype, pred_cov)  
    
  fit <- glm(formula,data=data,family=binomial)
  or <- exp(coef(fit))
  s <- summary(fit)
  beta <- s[["coefficients"]][2,1]
    
  se <- s[["coefficients"]][2,2]
  ci_lower <- exp(beta - 1.96 * se)
  ci_upper <- exp(beta + 1.96 * se)    
  
  result <- data.frame(predictor=predictor,
                       N=nrow(data),
                       OR=or[2],
                       CI_lower = ci_lower,
                       CI_upper = ci_upper,                       
                       beta=beta,
                       se=se,
                       z=s[["coefficients"]][2,3],
                       pval=s[["coefficients"]][2,4])
  return(result)
}

In [ ]:
n_pcs <- 10

pc_cols <- if (n_pcs > 0) paste0("PC", 1:n_pcs) else character(0)
pc_cols

In [ ]:
covariates <- c("Sex", 'Age', pc_cols)

In [ ]:
working_dir <- "/working_directory/polypharmacy_PGS"

In [ ]:
df_file <- glue("{working_dir}/polypharmacy_all_ancestry_multi_PGS.csv")

all_ancestry_df <- read.csv(df_file, stringsAsFactors = FALSE)
head(all_ancestry_df)

In [ ]:
prs_cols <- c( 
 'T1_anx2026_b37_PGS_PRScs',
 'T1_bip2025_b3738_PGS_PRScs',
 'T1_ed2025_b3738_PGS_PRScs',
 'T1_id2022_b3738_PGS_PRScs',
 'T1_mdd2025_b37_PGS_PRScs',
 'T1_ocd2025_b37_PGS_PRScs',
 'T1_ptsd2024_b37_PGS_PRScs',
 'T1_scz2025_PGS_PRScs',
 'T1_sud2023_b37_PGS_PRScs',
 'T2_extraversion2015_b37_PGS_PRScs',
 'T2_loneliness2025_b3738_PGS_PRScs',
 'T2_mtag2025_PGS_PRScs',
 'T2_neuroticism2025_b3738_PGS_PRScs',
 'T2_rtb2023_b3738_PGS_PRScs',
 'T2_swb2024_b3738_PGS_PRScs',
 'T3_cad2025_multi_b3738_PGS_PRScs',
 'T3_chronic_pain_2024_multi_b3738_PGS_PRScs',
 'T3_insomnia2024_multi_b3738_PGS_PRScs',
 'T3_migraine2025_eur_B3738_PGS_PRScs',
 'T3_t2d2025_b3738_PGS_PRScs',
 'T4_bmi2025_b3738_PGS_PRScs',
 'T4_crp2025_multi_b3738_PGS_PRScs',
 'T5_adhd2025_b3738_PGS_PRScs',
 'T5_asd2019_b37_PGS_PRScs',
 'T5_intelligence2025_multi_PGS_PRScs',
 'T5_neuro2025_eur_PGS_PRScs',
 'T5_ts2019_b37_PGS_PRScs'
)

In [ ]:
library(glue)
library(openxlsx)

phenotypes <- c("psychotropic_polypharmacy")

all_results <- list()   # store results

for (trait_prs in prs_cols) {
  for (phen in phenotypes) {
    
    result <- model2logistic(all_ancestry_df, trait_prs, phen, covariates)
    
    result$trait_prs <- trait_prs
    result$phenotype <- phen
    
    all_results[[trait_prs]] <- result
  }
}

# combine all results
final_results_df <- do.call(rbind, all_results)

out_put <- glue("{working_dir}/individual_PGS_polypharmacy_{polypharmacy_t}_{n_pcs}_pcs.xlsx")
write_xlsx(final_results_df, out_put)

In [ ]:
final_results_df$FDR <- p.adjust(final_results_df$pval, method = "fdr")
final_results_df$bonferroni <- p.adjust(final_results_df$pval, method = "bonferroni")

final_results_df$sig <- "Not significant"

final_results_df$sig[final_results_df$pval < 0.05] <- "Uncorrected"

final_results_df$sig[final_results_df$FDR < 0.05] <- "FDR corrected"
final_results_df$sig[final_results_df$bonferroni < 0.05] <- "Bonferroni corrected"

In [ ]:
print(final_results_df$predictor)

In [ ]:
final_results_df

In [ ]:
name_map <- c(
  "T1_anx2026_b37_PGS_PRScs"                  = "Anxiety disorder PGS",
  "T1_bip2025_b3738_PGS_PRScs"                = "Bipolar disorder PGS",
  "T1_ed2025_b3738_PGS_PRScs"                 = "Eating disorder PGS",
  "T1_id2022_b3738_PGS_PRScs"                 = "Internalizing disorder PGS",
  "T1_mdd2025_b37_PGS_PRScs"                  = "Major depressive disorder PGS",
  "T1_ocd2025_b37_PGS_PRScs"                  = "Obsessive-compulsive disorder PGS",
  "T1_ptsd2024_b37_PGS_PRScs"                 = "Post traumatic stress disorder PGS",
  "T1_scz2025_PGS_PRScs"                      = "Schizophrenia PGS" ,
  "T1_sud2023_b37_PGS_PRScs"                  = "Substance use disorder PGS",
                
  "T2_extraversion2015_b37_PGS_PRScs"         = "Extraversion PGS",
  "T2_loneliness2025_b3738_PGS_PRScs"         = "Loneliness PGS",
  "T2_mtag2025_PGS_PRScs"                     = "Educational attainment PGS",
  "T2_neuroticism2025_b3738_PGS_PRScs"        = "Neuroticism PGS",
  "T2_rtb2023_b3738_PGS_PRScs"                = "Risk taking PGS",
  "T2_swb2024_b3738_PGS_PRScs"                = "Subjective wellbeing PGS",
                
  "T3_cad2025_multi_b3738_PGS_PRScs"          = "Coronary artery disease PGS",
  "T3_chronic_pain_2024_multi_b3738_PGS_PRScs"= "Chronic pain PGS",
  "T3_insomnia2024_multi_b3738_PGS_PRScs"     = "Insomnia PGS",
  "T3_migraine2025_eur_B3738_PGS_PRScs"       = "Migraine PGS",
  "T3_t2d2025_b3738_PGS_PRScs"                = "Type 2 diabetes PGS",
  
  "T4_bmi2025_b3738_PGS_PRScs"                = "Body mass index PGS",
  "T4_crp2025_multi_b3738_PGS_PRScs"          = "C-reactive protein PGS",
                
  "T5_adhd2025_b3738_PGS_PRScs"               = "ADHD PGS",
  "T5_asd2019_b37_PGS_PRScs"                  = "Autism spectrum disorder PGS",
  "T5_intelligence2025_multi_PGS_PRScs"       = "Intelligence PGS",
  "T5_neuro2025_eur_PGS_PRScs"                = "Motor development PGS",
  "T5_ts2019_b37_PGS_PRScs"                   = "Tourette syndrome PGS"
)

In [ ]:
final_results_df$predictor <- name_map[final_results_df$predictor]

In [ ]:
# final_results_df
out_put <- glue("{working_dir}/individual_PGS_polypharmacy_all_ancestry_{polypharmacy_t}_age_sex_{n_pcs}_pcs.xlsx")
write_xlsx(final_results_df, out_put)


In [ ]:
library(ggplot2)

plot_p <- ggplot(final_results_df, aes(x = OR, y = reorder(predictor, OR))) +
  geom_point(aes(color = sig), size = 2) +
  geom_errorbarh(aes(xmin = CI_lower, xmax = CI_upper, color = sig), width = 0.5) +
  geom_vline(xintercept = 1, color = "black", linewidth = 0.4, linetype = "dashed") +
  scale_x_log10() +
  scale_color_manual(values = c(
    "ns" = "#bdbdbd",
    "Uncorrected" = "#4daf4a",
    "FDR corrected" = "#377eb8",
    "Bonferroni corrected" = "#e41a1c"
  )) +
  labs(
    x = "Odds ratio",
    y = "Polygenic score",
    color = "Significance (P < 0.05)"
  ) +
  #theme_minimal(base_size = 12) +
  guides(
    color = guide_legend(nrow = 2, 
                         byrow = TRUE,
                         keywidth = 0.3,
        keyheight = 0.3)
  ) +
  theme_bw(base_size = 6, base_family = "Arial") + 
  theme(
    axis.line.x = element_line(color = "black"),
    axis.line.y = element_blank(),
    legend.position = "bottom",
    legend.justification = "right",
    legend.box.just = "right",
    legend.margin = margin(t = 0, r = 0, l=0),
    legend.title = element_text(size = 5),
    legend.text = element_text(size = 5)      
  )

In [ ]:
ggsave(
  filename = glue("{working_dir}/individual_PGS_polypharmacy_{polypharmacy_t}_age_sex_{n_pcs}_pcs.pdf"),
  plot = plot_p,
  width = 70,
  height = 90,
  units = "mm",
  device = cairo_pdf
)
